In [2]:
!pip install rioxarray


In [3]:
pip install xarray netCDF4 rasterio


Note: you may need to restart the kernel to use updated packages.


In [7]:
import xarray as xr
import rioxarray
import os

# Ruta al archivo NetCDF
ruta_nc = r"C:/Users/paola/Tesis/02_Procesados/SST_monthly_detrended_1981_2025_0p5deg.nc"
ds = xr.open_dataset(ruta_nc)

# Carga y reordena la variable
sst = ds['sst_detrended'].transpose('time', 'lat', 'lon')

# Establece la proyección geográfica
sst.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
sst.rio.write_crs("EPSG:4326", inplace=True)

# Crear subcarpeta organizada
output_folder = r"C:/Users/paola/Tesis/02_Procesados/SST_tif_detrended"
os.makedirs(output_folder, exist_ok=True)

# Exportar los .tif mensuales
for i in range(sst.sizes['time']):
    t = sst.isel(time=i)
    fecha = str(t['time'].values)[:10]  # formato YYYY-MM-DD
    nombre_archivo = f"sst_detrended_{fecha}.tif"
    ruta_salida = os.path.join(output_folder, nombre_archivo)
    
    print(f"Guardando: {nombre_archivo}")
    t.rio.to_raster(ruta_salida)


Guardando: sst_detrended_1981-09-01.tif
Guardando: sst_detrended_1981-10-01.tif
Guardando: sst_detrended_1981-11-01.tif
Guardando: sst_detrended_1981-12-01.tif
Guardando: sst_detrended_1982-01-01.tif
Guardando: sst_detrended_1982-02-01.tif
Guardando: sst_detrended_1982-03-01.tif
Guardando: sst_detrended_1982-04-01.tif
Guardando: sst_detrended_1982-05-01.tif
Guardando: sst_detrended_1982-06-01.tif
Guardando: sst_detrended_1982-07-01.tif
Guardando: sst_detrended_1982-08-01.tif
Guardando: sst_detrended_1982-09-01.tif
Guardando: sst_detrended_1982-10-01.tif
Guardando: sst_detrended_1982-11-01.tif
Guardando: sst_detrended_1982-12-01.tif
Guardando: sst_detrended_1983-01-01.tif
Guardando: sst_detrended_1983-02-01.tif
Guardando: sst_detrended_1983-03-01.tif
Guardando: sst_detrended_1983-04-01.tif
Guardando: sst_detrended_1983-05-01.tif
Guardando: sst_detrended_1983-06-01.tif
Guardando: sst_detrended_1983-07-01.tif
Guardando: sst_detrended_1983-08-01.tif
Guardando: sst_detrended_1983-09-01.tif


In [12]:
import xarray as xr
import rioxarray
import os

# Ruta del NetCDF
ruta_nc = r"C:/Users/paola/Tesis/02_Procesados/rice_bestcluster.nc"
ds = xr.open_dataset(ruta_nc)

# Seleccionar la variable
rice = ds['rice_yield_bestcluster']  # (time, lat, lon)

# Renombrar dimensiones explícitamente para rioxarray (y, x)
rice = rice.rename({'lat': 'y', 'lon': 'x'})

# Asignar sistema de coordenadas y preparar para exportar
rice.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rice.rio.write_crs("EPSG:4326", inplace=True)

# Carpeta de salida
output_folder = r"C:/Users/paola/Tesis/02_Procesados/RICE_tif_bestcluster"
os.makedirs(output_folder, exist_ok=True)

# Exportar un archivo por año
for i in range(rice.sizes['time']):
    t = rice.isel(time=i)
    anio = str(t['time'].values)[:4]
    ruta_salida = os.path.join(output_folder, f"rice_yield_{anio}.tif")
    t.rio.to_raster(ruta_salida)

print("✅ ¡Todos los GeoTIFFs se exportaron correctamente!")


✅ ¡Todos los GeoTIFFs se exportaron correctamente!


In [9]:
import xarray as xr

# Ruta al archivo
ruta = r"C:/Users/paola/Tesis/02_Procesados/rice_bestcluster.nc"
ds = xr.open_dataset(ruta)

# Mostrar información general del dataset
print("\nResumen del dataset:")
print(ds)

# Listar variables disponibles
print("\nVariables disponibles:")
print(ds.data_vars)

# Listar coordenadas y dimensiones
print("\nCoordenadas:")
print(ds.coords)

print("\nDimensiones:")
print(ds.dims)



Resumen del dataset:
<xarray.Dataset> Size: 37MB
Dimensions:                 (lon: 720, lat: 360, time: 36)
Coordinates:
  * lon                     (lon) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
  * lat                     (lat) float64 3kB -89.75 -89.25 ... 89.25 89.75
  * time                    (time) datetime64[ns] 288B 1981-01-01 ... 2016-01-01
Data variables:
    rice_yield_bestcluster  (time, lat, lon) float32 37MB ...

Variables disponibles:
Data variables:
    rice_yield_bestcluster  (time, lat, lon) float32 37MB ...

Coordenadas:
Coordinates:
  * lon      (lon) float64 6kB 0.25 0.75 1.25 1.75 ... 358.2 358.8 359.2 359.8
  * lat      (lat) float64 3kB -89.75 -89.25 -88.75 -88.25 ... 88.75 89.25 89.75
  * time     (time) datetime64[ns] 288B 1981-01-01 1982-01-01 ... 2016-01-01

Dimensiones:
FrozenMappingWarningOnValuesAccess({'lon': 720, 'lat': 360, 'time': 36})


In [13]:
import rasterio
import numpy as np
import os
import glob

# Carpeta donde están los .tif exportados
carpeta = r"C:/Users/paola/Tesis/02_Procesados/RICE_tif_bestcluster"
archivos = sorted(glob.glob(os.path.join(carpeta, "*.tif")))

for path in archivos:
    with rasterio.open(path) as src:
        data = src.read(1)  # leer primera (y única) banda
        transform = src.transform
        nodata = src.nodata or np.nan
        mask = ~np.isnan(data) if np.isnan(nodata) else data != nodata
        
        # Coordenadas geográficas de una muestra de pixeles válidos
        coords = [src.xy(*ix[::-1]) for ix in np.argwhere(mask)[:3]]

        print(f"\n🗂️ Archivo: {os.path.basename(path)}")
        print(f"📐 Tamaño: {data.shape}")
        print(f"✅ Pixeles válidos: {np.count_nonzero(mask)}")
        print(f"🔢 Valores: min = {np.nanmin(data)}, max = {np.nanmax(data)}")
        print(f"📍 Muestra de coordenadas válidas: {coords}")



🗂️ Archivo: rice_yield_1981.tif
📐 Tamaño: (360, 720)
✅ Pixeles válidos: 3186
🔢 Valores: min = -8.069526672363281, max = 18.37626838684082
📍 Muestra de coordenadas válidas: [(np.float64(58.25), np.float64(217.75)), (np.float64(58.75), np.float64(217.25)), (np.float64(59.75), np.float64(218.25))]

🗂️ Archivo: rice_yield_1982.tif
📐 Tamaño: (360, 720)
✅ Pixeles válidos: 3186
🔢 Valores: min = -2.74141001701355, max = 4.488324165344238
📍 Muestra de coordenadas válidas: [(np.float64(58.25), np.float64(217.75)), (np.float64(58.75), np.float64(217.25)), (np.float64(59.75), np.float64(218.25))]

🗂️ Archivo: rice_yield_1983.tif
📐 Tamaño: (360, 720)
✅ Pixeles válidos: 3186
🔢 Valores: min = -2.3989157676696777, max = 7.971563339233398
📍 Muestra de coordenadas válidas: [(np.float64(58.25), np.float64(217.75)), (np.float64(58.75), np.float64(217.25)), (np.float64(59.75), np.float64(218.25))]

🗂️ Archivo: rice_yield_1984.tif
📐 Tamaño: (360, 720)
✅ Pixeles válidos: 3186
🔢 Valores: min = -3.02714204788

In [14]:
import rasterio
import glob
import os

# Carpeta donde tienes los .tif
carpeta = r"C:/Users/paola/Tesis/02_Procesados/RICE_tif_bestcluster"
archivos = sorted(glob.glob(os.path.join(carpeta, "*.tif")))

for path in archivos:
    with rasterio.open(path) as src:
        print(f"\n🗂️ {os.path.basename(path)}")
        print(f"  Bandas: {src.count}")
        print(f"  Dimensiones: {src.width}x{src.height}")
        print(f"  CRS: {src.crs}")



🗂️ rice_yield_1981.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1982.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1983.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1984.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1985.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1986.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1987.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1988.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1989.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1990.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1991.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1992.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yield_1993.tif
  Bandas: 1
  Dimensiones: 720x360
  CRS: EPSG:4326

🗂️ rice_yie